In [4]:
import pandas as pd
import uuid
import mlflow
import argparse    

In [5]:
def generate_uuids(n: int):
    ride_ids = []
    for i in range(n):
        ride_ids.append(str(uuid.uuid1()))
    return ride_ids

def read_dataframe(filename: str) -> pd.DataFrame:
    df = pd.read_parquet(filename)
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df['duration'] = df['duration'].dt.total_seconds() / 60
    df = df[(df['duration'] >= 1) & (df['duration'] <= 60)]
    
    return df

def prepare_dicts(df: pd.DataFrame) -> dict:
    cat = ['PULocationID', 'DOLocationID']
    df[cat] = df[cat].astype(str)
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    cat = ['PU_DO']
    num = ['trip_distance']
    dicts = df[cat + num].to_dict(orient='records')

    return dicts

In [6]:
def edit_doc(df: pd.DataFrame, pred) -> pd.DataFrame:
    doc = pd.DataFrame()
    
    doc['ride_id'] = generate_uuids(len(df))
    doc['PULocationID'] = df['PULocationID']
    doc['DOLocationID'] = df['DOLocationID']
    doc['predict_duration'] = pred
    doc['actual_duration'] = df['duration']
    doc['diff'] = doc['actual_duration'] - doc['predict_duration']

    return doc

In [ ]:

def run():
    parser = argparse.ArgumentParser(description='predict the duration')

    parser.add_argument('--taxi_type', type=str, required=True, help='choose the taxi type you will predict')
    parser.add_argument('--year', type=int, required=True, help='choose the year you will predict')
    parser.add_argument('--month', type=int, required=True, help='choose the month you will predict')

    args = parser.parse_args()

    input_path = f'https://d37ci6vzurychx.cloudfront.net/trip-data/{args.taxi_type}_tripdata_{args.year}-{args.month:02}.parquet'
    output_path = f'output/{args.taxi_type}/{args.year}-{args.month:02}.parquet'

    df = read_dataframe(input_path)
    dicts = prepare_dicts(df)

    model_uri = 'runs:/2c2a1858070a407283f4e07cc29dcaed/model'
    model = mlflow.pyfunc.load_model(model_uri)

    y_pred = model.predict(dicts)
    doc = edit_doc(df, y_pred)
    doc.to_parquet(path=output_path, index=False)
    